# Grundlegenden Datensatzanalyse

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
df = pd.read_csv("./hubway/sample_data/201501-hubway-tripdata.csv")
display(df.head(3))
df.info()


,tripduration,starttime,stoptime,start station id,start station name,start station latitude,start station longitude,end station id,end station name,end station latitude,end station longitude,bikeid,usertype,birth year,gender
0,542,2015-01-01 00:21:44,2015-01-01 00:30:47,115,Porter Square Station,42.387995,-71.119084,96,Cambridge Main Library at Broadway / Trowbridg...,42.373379,-71.111075,277,Subscriber,1984,1
1,438,2015-01-01 00:27:03,2015-01-01 00:34:21,80,MIT Stata Center at Vassar St / Main St,42.361962,-71.092053,95,Cambridge St - at Columbia St / Webster Ave,42.372969,-71.094445,648,Subscriber,1985,1
2,254,2015-01-01 00:31:31,2015-01-01 00:35:46,91,One Kendall Square at Hampshire St / Portland St,42.366277,-71.091690,68,Central Square at Mass Ave / Essex St,42.365070,-71.103100,555,Subscriber,1974,1


<class 'pandas.DataFrame'>
RangeIndex: 7840 entries, 0 to 7839
Data columns (total 15 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   tripduration             7840 non-null   int64  
 1   starttime                7840 non-null   str    
 2   stoptime                 7840 non-null   str    
 3   start station id         7840 non-null   int64  
 4   start station name       7840 non-null   str    
 5   start station latitude   7840 non-null   float64
 6   start station longitude  7840 non-null   float64
 7   end station id           7840 non-null   int64  
 8   end station name         7840 non-null   str    
 9   end station latitude     7840 non-null   float64
 10  end station longitude    7840 non-null   float64
 11  bikeid                   7840 non-null   int64  
 12  usertype                 7840 non-null   str    
 13  birth year               7840 non-null   str    
 14  gender                   7840 non-n

inspect further trip time

In [42]:
calc_trip_time = (
    pd.to_datetime(df["stoptime"]) - pd.to_datetime(df["starttime"])
).dt.total_seconds()

diff = calc_trip_time - df["tripduration"]

print("Abweichungen:", (diff != 0).sum())
print("Durchschnittliche Abweichung:", diff.abs().mean(), "Sekunden")


Abweichungen: 3946
Durchschnittliche Abweichung: 0.5033163265306122 Sekunden


Der kleine Mittelwert allein reicht nicht als Beleg. Maximum und Verteilung der Abweichungen werden unten geprueft; es werden keine Fahrten automatisch entfernt.

## Fahrten zur gleichen Station
Stations-IDs statt Namen vergleichen. Rundfahrten bleiben erhalten, auch fuer die Entfernungsauswertung. Eine Luftlinie von null bedeutet nicht, dass keine Strecke gefahren wurde.

In [43]:
same_station = df["start station id"].notna() & df["end station id"].notna() & df["start station id"].eq(df["end station id"])
same_station_trips = df.loc[same_station]
print("Anzahl der Fahrten, bei denen Start- und Endstation gleich sind:", len(same_station_trips))
print("Durchschnittliche Fahrtdauer bei diesen Fahrten:", same_station_trips["tripduration"].mean(), "Sekunden")
print("Anazhal der Fahrten über 5min:", (same_station_trips["tripduration"] > 300).sum())

Anzahl der Fahrten, bei denen Start- und Endstation gleich sind: 142
Durchschnittliche Fahrtdauer bei diesen Fahrten: 1774.2605633802816 Sekunden
Anazhal der Fahrten über 5min: 131


Die Ausgabe beschreibt moegliche Rundfahrten, nicht bewiesene Fehler.

## 1. Fehlende Werte und Zeitstempel
Nur pruefen, nichts loeschen: df und die CSV bleiben unveraendert. Platzhalter werden fuer die Pruefung normalisiert. Der erwartete Fahrtbeginn ist Januar 2015; ein Fahrtende im Februar ist nicht automatisch falsch.

In [44]:
text = df.astype('string').apply(lambda col: col.str.strip())
placeholders = {'', 'null', 'none', 'nan', 'na', 'n/a', '\\n', 'unknown', 'unbekannt'}
missing = df.isna() | text.apply(lambda col: col.str.lower().isin(placeholders))
display(pd.DataFrame({'Nullwerte': df.isna().sum(), 'Fehlend inkl. Platzhalter': missing.sum()}))

start = pd.to_datetime(text['starttime'].mask(missing['starttime']), format='%Y-%m-%d %H:%M:%S', errors='coerce')
end = pd.to_datetime(text['stoptime'].mask(missing['stoptime']), format='%Y-%m-%d %H:%M:%S', errors='coerce')
duration = pd.to_numeric(text['tripduration'].mask(missing['tripduration']), errors='coerce')
bad_month = start.notna() & start.dt.to_period('M').ne(pd.Period('2015-01'))
time_checks = pd.Series({
    'Start fehlt oder ist unlesbar': int(start.isna().sum()),
    'Ende fehlt oder ist unlesbar': int(end.isna().sum()),
    'Fahrtbeginn ausserhalb Januar 2015': int(bad_month.sum()),
    'Ende vor Beginn': int((end < start).sum()),
}, name='Anzahl')
display(time_checks.to_frame())
display(df.loc[start.isna() | end.isna() | bad_month | (end < start)].head(10))


,Nullwerte,Fehlend inkl. Platzhalter
tripduration,0,0
starttime,0,0
stoptime,0,0
start station id,0,0
start station name,0,0
start station latitude,0,0
start station longitude,0,0
end station id,0,0
end station name,0,0
end station latitude,0,0


,Anzahl
Start fehlt oder ist unlesbar,0
Ende fehlt oder ist unlesbar,0
Fahrtbeginn ausserhalb Januar 2015,0
Ende vor Beginn,0


,tripduration,starttime,stoptime,start station id,start station name,start station latitude,start station longitude,end station id,end station name,end station latitude,end station longitude,bikeid,usertype,birth year,gender


Nullwerte allein reichen nicht: Die erste Tabelle zaehlt auch erkannte Textplatzhalter. Die zweite unterscheidet unlesbare Zeiten, falschen Startmonat und negative Zeitfolgen. Die Beispieltabelle zeigt bis zu zehn betroffene Fahrten. Zeitstempel werden als lokale Quellzeiten interpretiert, nicht nach UTC verschoben.

## 2. Geburtsjahr, Alter und Geschlechtscodes
Alter = Fahrtjahr minus Geburtsjahr. Alter ueber 100 ist hier nur eine Untersuchungsmarkierung, keine Loeschregel. Code 0 bedeutet unbekannt, 1 maennlich, 2 weiblich; weitere Codes separat ausweisen.

In [45]:
birth = pd.to_numeric(text['birth year'].mask(missing['birth year']), errors='coerce')
birth_finite = (birth.notna() & np.isfinite(birth)).fillna(False)
birth_integer = (birth_finite & birth.mod(1).eq(0)).fillna(False)
age = start.dt.year - birth.where(birth_integer)
birth_bad = ~missing['birth year'] & ~birth_integer
gender = pd.to_numeric(text['gender'].mask(missing['gender']), errors='coerce')
gender_bad = (~missing['gender'] & ~gender.isin([0, 1, 2])).fillna(False)
demo_checks = pd.Series({
    'Geburtsjahr fehlt / Platzhalter': int(missing['birth year'].sum()),
    'Geburtsjahr nicht als endliche Ganzzahl lesbar': int(birth_bad.sum()),
    'Alter nicht berechenbar': int(age.isna().sum()),
    'Geschlecht fehlt / Platzhalter': int(missing['gender'].sum()),
    'Geschlechtscode 0 (unbekannt)': int(gender.eq(0).sum()),
    'Unerwarteter Geschlechtswert': int(gender_bad.sum()),
}, name='Anzahl')
display(demo_checks.to_frame())
display(text['gender'].value_counts(dropna=False).rename('Fahrten').to_frame())
display(age.describe().rename('Alter in Jahren').to_frame())
display(df.loc[birth_bad | age.lt(0).fillna(False) | age.gt(100).fillna(False) | gender_bad, ['starttime', 'birth year', 'gender']].head(10))


,Anzahl
Geburtsjahr fehlt / Platzhalter,254
Geburtsjahr nicht als endliche Ganzzahl lesbar,0
Alter nicht berechenbar,254
Geschlecht fehlt / Platzhalter,0
Geschlechtscode 0 (unbekannt),254
Unerwarteter Geschlechtswert,0


,Fahrten
gender,
1,5672
2,1914
0,254


,Alter in Jahren
count,7586.0
mean,34.244793
std,10.344051
min,19.0
25%,27.0
50%,31.0
75%,38.0
max,115.0


,starttime,birth year,gender
6774,2015-01-25 02:36:20,1900,1


Die Haeufigkeitstabelle zeigt die tatsaechlichen Geschlechtswerte, einschliesslich Platzhaltern. Fehlende Demografie macht eine Fahrt nicht insgesamt unbrauchbar: Fuer die betroffenen KPI-Kategorien spaeter Unbekannt verwenden. Auffaellige Alterswerte vor einer Grenzfestlegung untersuchen.

## 3. Dauer und Abweichung zur Zeitdifferenz
Dauer in Sekunden. Nur endliche Zahlen vergleichen; fehlende oder unlesbare Angaben gesondert zaehlen. Die zehn laengsten Fahrten werden angezeigt, aber nicht entfernt.

In [53]:
duration_finite = (duration.notna() & np.isfinite(duration)).fillna(False)
duration_clean = duration.where(duration_finite)
elapsed = (end - start).dt.total_seconds()
duration_diff = elapsed - duration_clean
duration_checks = pd.Series({
    'Dauer fehlt / Platzhalter': int(missing['tripduration'].sum()),
    'Dauer unlesbar oder nicht endlich': int((~missing['tripduration'] & ~duration_finite).sum()),
    'Dauer kleiner oder gleich null': int(duration_clean.le(0).sum()),
    'Dauer unter 60 Sekunden (Pruefung Quellfilter)': int(duration_clean.lt(60).sum()),
    'Zeitdifferenz und Dauer nicht vergleichbar': int(duration_diff.isna().sum()),
}, name='Anzahl')
display(duration_checks.to_frame())
print('Maximale Abweichung (Sekunden):', duration_diff.abs().max())
print('Mittlere Abweichung (Sekunden):', duration_diff.abs().mean())



,Anzahl
Dauer fehlt / Platzhalter,0
Dauer unlesbar oder nicht endlich,0
Dauer kleiner oder gleich null,0
Dauer unter 60 Sekunden (Pruefung Quellfilter),0
Zeitdifferenz und Dauer nicht vergleichbar,0


Maximale Abweichung (Sekunden): 1.0
Mittlere Abweichung (Sekunden): 0.5033163265306122


Maximum und Haeufigkeiten zeigen, ob hinter dem kleinen mittleren Unterschied einzelne grosse Abweichungen stecken. Lange Fahrten sind Untersuchungsfaelle, keine automatisch bewiesenen Fehler. NaN bei Kennzahlen bedeutet: Es liegen keine gueltigen Vergleichswerte vor.

## 4. Koordinaten und vollstaendige Duplikate
Breitengrad muss zwischen -90 und 90, Laengengrad zwischen -180 und 180 liegen. Gueltige Wertebereiche beweisen noch nicht die korrekte Lage einer Station. Duplikate beziehen sich auf alle Originalspalten, nicht allein auf Fahrrad oder Station.

In [47]:
coord_limits = {
    'start station latitude': (-90, 90),
    'start station longitude': (-180, 180),
    'end station latitude': (-90, 90),
    'end station longitude': (-180, 180),
}
coords = pd.DataFrame(index=df.index)
coord_valid = pd.DataFrame(index=df.index)
coord_rows = []
for column, (lower, upper) in coord_limits.items():
    values = pd.to_numeric(text[column].mask(missing[column]), errors='coerce')
    finite = (values.notna() & np.isfinite(values)).fillna(False)
    valid = (finite & values.between(lower, upper)).fillna(False)
    coords[column] = values
    coord_valid[column] = valid
    coord_rows.append({
        'Spalte': column,
        'Fehlend / Platzhalter': int(missing[column].sum()),
        'Unlesbar / nicht endlich': int((~missing[column] & ~finite).sum()),
        'Ausserhalb Wertebereich': int((finite & ~valid).sum()),
    })
display(pd.DataFrame(coord_rows).set_index('Spalte'))
bad_coords = ~coord_valid.all(axis=1)
same_coords = (coord_valid.all(axis=1)
               & coords['start station latitude'].eq(coords['end station latitude'])
               & coords['start station longitude'].eq(coords['end station longitude'])).fillna(False)
duplicate_extra = df.duplicated(keep='first')
duplicate_all = df.duplicated(keep=False)
print('Fahrten mit ungueltigen Koordinaten:', int(bad_coords.sum()))
print('Fahrten mit exakt gleichen gueltigen Start-/Zielkoordinaten:', int(same_coords.sum()))
print('Zusaetzliche vollstaendige Duplikatzeilen:', int(duplicate_extra.sum()))
print('Alle Zeilen in Duplikatgruppen:', int(duplicate_all.sum()))
display(df.loc[bad_coords].head(10))
display(df.loc[duplicate_all].sort_values(list(df.columns)).head(10))


,Fehlend / Platzhalter,Unlesbar / nicht endlich,Ausserhalb Wertebereich
Spalte,,,
start station latitude,0,0,0
start station longitude,0,0,0
end station latitude,0,0,0
end station longitude,0,0,0


Fahrten mit ungueltigen Koordinaten: 0
Fahrten mit exakt gleichen gueltigen Start-/Zielkoordinaten: 142
Zusaetzliche vollstaendige Duplikatzeilen: 0
Alle Zeilen in Duplikatgruppen: 0


,tripduration,starttime,stoptime,start station id,start station name,start station latitude,start station longitude,end station id,end station name,end station latitude,end station longitude,bikeid,usertype,birth year,gender


,tripduration,starttime,stoptime,start station id,start station name,start station latitude,start station longitude,end station id,end station name,end station latitude,end station longitude,bikeid,usertype,birth year,gender


### Q&A
Die Pruefungen decken fehlende Werte, Demografie, Zeiten, Dauer, Koordinaten und vollstaendige Duplikate ab. Sie veraendern keine Originaldaten. Die neuen Zellen sind noch auszufuehren; vorab werden keine Ergebnisse behauptet.

### Data Analysis Key Findings
- Nach dem Ausfuehren die konkreten Zaehler, maximale Dauerabweichung und auffaelligen Beispiele hier dokumentieren.
- Das Histogramm zeigt die Verteilung der positiven Dauerwerte einschliesslich langer Fahrten; es begruendet allein keine Ausschlussgrenze.

### Insights or Next Steps
- Auffaellige Beispiele pruefen und Regeln pro KPI dokumentieren; fehlende Demografie als Unbekannt behandeln und Rundfahrten behalten.
- Anschliessend einen kleinen kontrollierten Testdatensatz mit unabhaengigen Soll-Ergebnissen erstellen und die Regeln in die PySpark-Pipeline uebernehmen.

# stationenen
Es gibt zwei DAtein mit stationen eine ältere und eine neuere, überprüft werdens oll ob nur stationen hinzugekommen oder auch entfernt wurden

In [48]:
df_2016 = pd.read_csv("./hubway/sample_data/Hubway_Stations_2011_2016.csv")
df_2017 = pd.read_csv("./hubway/sample_data/Hubway_Stations_as_of_July_2017.csv")


display(df_2016.head(2))
display(df_2017.head(2))


,Station,Station ID,Latitude,Longitude,Municipality,# of Docks
0,Fan Pier,A32000,42.353287,-71.044389,Boston,15
1,Union Square - Brighton Ave. at Cambridge St.,A32001,42.353334,-71.137313,Boston,15


,Station ID,Station,Latitude,Longitude,Municipality,# of Docks
0,C32015,Dudley Town Common - Mt Pleasant Ave at Blue H...,42.325333,-71.075354,Boston,15
1,D32030,Faneuil St at Market St,42.354802,-71.150250,Boston,15


In [51]:
id_2016 = df_2016["Station ID"].astype("string").str.strip()
id_2017 = df_2017["Station ID"].astype("string").str.strip()

removed_in_2017 = df_2016.loc[~id_2016.isin(id_2017)]
added_in_2017 = df_2017.loc[~id_2017.isin(id_2016)]

print("Removed in 2017:")
display(removed_in_2017)
print("Added in 2017:")
display(added_in_2017)

Removed in 2017:


,Station,Station ID,Latitude,Longitude,Municipality,# of Docks


Added in 2017:


,Station ID,Station,Latitude,Longitude,Municipality,# of Docks
132,K32006,Commonwealth Ave At Babcock St,42.351547,-71.121263,Brookline,19
172,M32045,Museum of Science,42.367690,-71.071163,Cambridge,19
177,M32050,Verizon Innovation Hub 10 Ware Street,42.372509,-71.113054,Cambridge,19
178,M32051,Fresh Pond Reservation,42.382678,-71.143479,Cambridge,19
179,M32052,Cambridge Dept. of Public Works -147 Hampshire...,42.371197,-71.097599,Cambridge,19
192,Silber Way,Silber Way,42.349496,-71.100576,Brookline,19


====> dataset from 2016 can be ignored

analysiere genauer hubway data

In [54]:
df_2017.info()
df_2017.describe()

<class 'pandas.DataFrame'>
RangeIndex: 193 entries, 0 to 192
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Station ID    193 non-null    str    
 1   Station       193 non-null    str    
 2   Latitude      193 non-null    float64
 3   Longitude     193 non-null    float64
 4   Municipality  193 non-null    str    
 5   # of Docks    193 non-null    int64  
dtypes: float64(2), int64(1), str(3)
memory usage: 9.2 KB


,Latitude,Longitude,# of Docks
count,193.000000,193.000000,193.000000
mean,42.354947,-71.086062,17.544041
std,0.021905,0.031575,4.355238
min,42.303469,-71.166367,11.000000
25%,42.341814,-71.109420,15.000000
50%,42.354979,-71.085347,15.000000
75%,42.370677,-71.062679,19.000000
max,42.404490,-71.006212,46.000000
